# Final Rerun — UNM (paper evidence)


# UNM Final Rerun (paper evidence)

**Purpose**: clean, reproducible final rerun of the UNM experiment suite for the paper. This notebook is **not** an ablation — the final configuration has already been decided. Results here are the ones reported.

**Output root**: `runs_final_v1/` (siblings: historical `runs/`, `runs_lambda_ablation/`). These are separate on purpose — **do not overwrite the historical `runs/` directory**.

**Final configuration (all runs):**
- `lambda_u = 0.05` (decided by the λ ablation in `05_unm_lambda_ablation.ipynb`)
- `semi_start_epoch = 15` (UNM schedule)
- `patience_es = 40`
- `tau = 0.95`, `ema_decay = 0.99`, `semi_warmup_epochs = 20`
- `save_preds_vis = False` (inline figures disabled — use the post-hoc cell for viz)
- Architecture, backbone, preprocessing, batch size, augmentations: **identical** to the historical `runs/`

**Experiment catalog (27 per seed × 3 seeds = 81 runs):**
- 1× supervised
- 6× PL temporal (`semi_r{3,5,7,10,15,20}`)
- 6× PL random matched (`semi_std_matched_r{3,5,7,10,15,20}`)
- 1× PL all-lateral (`semi_all_lateral`)
- 6× MT temporal (`mean_teacher_r{3,5,7,10,15,20}`)
- 6× MT random matched (`mean_teacher_std_matched_r{3,5,7,10,15,20}`)
- 1× MT all-lateral (`mean_teacher_all_lateral`)

**Execution order**: seed_0 (27) → seed_1 (27) → seed_2 (27). After each seed block there is a read-only summary cell that shows which runs completed, best_epoch, and whether the best checkpoint is post-SSL.

**Skip logic** (per cell, 3 cases):
- **Case A (skip)**: `best_model.pt` + `test_metrics.csv` exist → skip training & eval
- **Case B (eval only)**: `best_model.pt` exists but `test_metrics.csv` or `*_run_report.json` missing → re-run eval only
- **Case C (train)**: `best_model.pt` missing → full training

**NOTE**: ablation notebooks 05 and 06 and the historical notebooks 01 and 04b remain untouched. The final-rerun output goes to `runs_final_v1/`, a brand-new root. Old runs are historical and must not be overwritten.


In [ ]:
# ============================================================
# SETUP — run this cell once after every runtime restart
# Do NOT run training here. Select one experiment cell below.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

# --- Environment fingerprint (for reproducibility debugging) ---
import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!pip show albumentations | grep Version
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"
# --------------------------------------------------------------

import sys
sys.path.append("/content/tesis-seg")

from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint
from src.visualization import show_dataset_examples, show_predictions

# ── Debug fingerprint helpers ─────────────────────────────────
import json, os, platform, subprocess, time, importlib.metadata

def _fp_get_version(pkg):
    try: return importlib.metadata.version(pkg)
    except Exception: return None

def _fp_git_hash_from_src(src_train_file):
    # Derive repo root from src/train.py: {repo_root}/src/train.py
    repo_root = os.path.dirname(os.path.dirname(os.path.abspath(src_train_file)))
    try:
        return subprocess.check_output(
            ["git", "log", "--oneline", "-1"], cwd=repo_root, stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception: return None

def _fp_save(pre, post, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump({"pre_run": pre, "post_run": post}, f, indent=2, default=str)

def _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                    unlabeled_ds=None, temporal_unlab_ds=None):
    import torch, sys
    from src.models import create_model

    # 1. Environment
    try: gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a"
    except: gpu_name = "n/a"
    env = {
        "python":      sys.version,
        "torch":       torch.__version__,
        "torchvision": _fp_get_version("torchvision"),
        "cuda":        torch.version.cuda,
        "cudnn":       str(torch.backends.cudnn.version()) if torch.cuda.is_available() else "n/a",
        "segmentation_models_pytorch": _fp_get_version("segmentation-models-pytorch"),
        "albumentations": _fp_get_version("albumentations"),
        "platform":    platform.platform(),
        "gpu_name":    gpu_name,
    }

    # 2. Code provenance — git hash derived from actual runtime src path
    import src.train, src.datasets, src.defaults, src.evaluate
    provenance = {
        "src_train":    src.train.__file__,
        "src_datasets": src.datasets.__file__,
        "src_defaults": src.defaults.__file__,
        "src_evaluate": src.evaluate.__file__,
        "git_hash":     _fp_git_hash_from_src(src.train.__file__),
    }

    # 3. Effective config
    cfg_keys = [
        "seed", "arch", "backbone", "n_classes",
        "image_preproc", "mask_smoothing", "target_size", "use_pad", "imagenet_norm",
        "batch_size", "num_workers", "drop_last", "num_augmented",
        "lr", "weight_decay", "epochs", "warmup_epochs", "patience_es", "eval_threshold",
        "use_semi", "use_temp_consistency",
        "lambda_u", "tau", "ema_decay", "semi_start_epoch", "semi_warmup_epochs", "lambda_t",
        "unlabeled_subdir", "exp_dir",
    ]
    eff_cfg = {k: cfg.get(k) for k in cfg_keys}
    unlab_loader = loaders.get("unlabeled_loader")
    eff_cfg["batch_size_unlab"] = unlab_loader.batch_size if unlab_loader is not None else None

    # 4. Dataset / loader facts
    ds_facts = {
        "len_train_ds":          len(train_ds),
        "len_val_ds":            len(val_ds),
        "len_test_ds":           len(test_ds),
        "len_unlabeled_ds":      len(unlabeled_ds) if unlabeled_ds is not None else None,
        "len_temporal_unlab_ds": len(temporal_unlab_ds) if temporal_unlab_ds is not None else None,
        "train_loader_batch_size":      loaders["train_loader"].batch_size,
        "train_loader_num_workers":     loaders["train_loader"].num_workers,
        "train_loader_drop_last":       loaders["train_loader"].drop_last,
        "unlabeled_loader_batch_size":  unlab_loader.batch_size if unlab_loader else None,
        "unlabeled_loader_num_workers": unlab_loader.num_workers if unlab_loader else None,
        "unlabeled_loader_drop_last":   unlab_loader.drop_last if unlab_loader else None,
    }

    # 5. Sample identifiers — reads .files attribute, no IO beyond what dataset already did
    try: sup_ids = train_ds.files[:5]
    except Exception as e: sup_ids = f"unavailable: {e}"
    try: unl_ids = unlabeled_ds.files[:5] if unlabeled_ds is not None else None
    except Exception as e: unl_ids = f"unavailable: {e}"
    sample_ids = {"first5_train": sup_ids, "first5_unlabeled": unl_ids}

    # 6. Batch tensor shapes — analytical, no DataLoader consumed, no RNG touched
    try:
        H, W = cfg["target_size"]
        C = 3  # IMREAD_COLOR: grayscale PNGs expand to 3 identical channels
        eff_bs = cfg["batch_size"] * (1 + cfg.get("num_augmented", 0))  # flatten_collate
        bs_u = max(1, cfg["batch_size"] // 4)  # mirrors datasets.py build_dataloaders
        batch_shapes = {
            "xb":   [eff_bs, C, H, W],
            "yb":   [eff_bs, 1, H, W],
            "xw_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "xs_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "note": "analytically derived from cfg — no DataLoader consumed",
        }
    except Exception as e:
        batch_shapes = {"error": str(e)}

    # 7. Model fingerprint — RNG save/restore so training is unaffected.
    # Belt-and-suspenders: run_training() also calls seed_everything(seed) first.
    try:
        import random as _random, numpy as _np
        _rng = {
            "py":   _random.getstate(),
            "np":   _np.random.get_state(),
            "th":   torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        model_fp = {
            "total_params":          sum(p.numel() for p in _m.parameters()),
            "trainable_params":      sum(p.numel() for p in _m.parameters() if p.requires_grad),
            "first_state_dict_keys": list(_m.state_dict().keys())[:8],
        }
        del _m
        _random.setstate(_rng["py"])
        _np.random.set_state(_rng["np"])
        torch.set_rng_state(_rng["th"])
        if _rng["cuda"] is not None:
            torch.cuda.set_rng_state_all(_rng["cuda"])
    except Exception as e:
        model_fp = {"error": str(e)}

    return {
        "timestamp_utc":     time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "environment":       env,
        "provenance":        provenance,
        "effective_cfg":     eff_cfg,
        "dataset_facts":     ds_facts,
        "sample_ids":        sample_ids,
        "batch_shapes":      batch_shapes,
        "model_fingerprint": model_fp,
    }


def _fp_collect_post(artifacts, results):
    history = artifacts.get("history") or []
    best_row = max(history, key=lambda r: r.get("val_iou_global", 0.0)) if history else None
    vm = (results or {}).get("val_metrics", {})
    tm = (results or {}).get("test_metrics", {})
    return {
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "best_path":     artifacts.get("best_path"),
        "best_epoch_info": {
            "epoch":          best_row.get("epoch") if best_row else None,
            "val_iou_global": best_row.get("val_iou_global") if best_row else None,
            "val_loss":       best_row.get("val_loss") if best_row else None,
        },
        "val_metrics": {
            "f1_global":       vm.get("global_f1"),
            "iou_global":      vm.get("global_iou"),
            "f1_sample_mean":  vm.get("sample_mean_f1"),
            "iou_sample_mean": vm.get("sample_mean_iou"),
        },
        "test_metrics": {
            "f1_global":       tm.get("global_f1"),
            "iou_global":      tm.get("global_iou"),
            "f1_sample_mean":  tm.get("sample_mean_f1"),
            "iou_sample_mean": tm.get("sample_mean_iou"),
        },
        "finished_successfully": True,
        "exception": None,
    }
# ──────────────────────────────────────────────────────────────

print("Imports OK — select one experiment cell below and run it.")

In [ ]:
# === ENVIRONMENT SANITY CHECK (UNM) ===
# Run this ONCE after the setup cell. Verifies Drive access, data dirs,
# GPU, key packages, and creates the output root. Raises on any failure
# so training never starts against a half-wired environment.
import os, sys

_errors = []
_warnings = []

# 1. Google Drive mount
_drive_root = "/content/drive/MyDrive"
if not os.path.isdir(_drive_root):
    _errors.append(f"Google Drive not mounted at {_drive_root}")

# 2. Data directories (img_root + unlabeled pool parents)
_img_root = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
if not os.path.isdir(_img_root):
    _errors.append(f"img_root missing: {_img_root}")

_pool_parents = [
    "unlabeling_r3_max0",
    "unlabeling_r5_max0",
    "unlabeling_r7_max0",
    "unlabeling_r10_max0",
    "unlabeling_r15_max0",
    "unlabeling_r20_max0",
    "unlabeling_std_matched_r3",
    "unlabeling_std_matched_r5",
    "unlabeling_std_matched_r7",
    "unlabeling_std_matched_r10",
    "unlabeling_std_matched_r15",
    "unlabeling_std_matched_r20",
    "unlabeling_all_lateral",
]
_missing_pools = []
for _p in _pool_parents:
    _full = os.path.join(_img_root, _p)
    if not os.path.isdir(_full):
        _missing_pools.append(_p)
if _missing_pools:
    _errors.append(f"{len(_missing_pools)} unlabeled pool dirs missing under {_img_root}: {_missing_pools}")

# 3. GPU
try:
    import torch
    if not torch.cuda.is_available():
        _errors.append("CUDA not available — training will fail or be unusable")
    else:
        print(f"[GPU] {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda} | torch {torch.__version__}")
except Exception as _e:
    _errors.append(f"torch import failed: {_e}")

# 4. Key packages (import + print version)
_pkgs = [("segmentation_models_pytorch", "smp"), ("albumentations", "A")]
for _pkg, _alias in _pkgs:
    try:
        _m = __import__(_pkg)
        _v = getattr(_m, "__version__", "unknown")
        print(f"[pkg] {_pkg}: {_v}")
    except Exception as _e:
        _errors.append(f"cannot import {_pkg}: {_e}")

# 5. Create output root (non-destructive; exist_ok=True)
_output_root = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1"
try:
    os.makedirs(_output_root, exist_ok=True)
    print(f"[output] root ready: {_output_root}")
except Exception as _e:
    _errors.append(f"cannot create output root {_output_root}: {_e}")

if _errors:
    print("\n*** ENVIRONMENT SANITY CHECK FAILED ***")
    for _err in _errors:
        print(f"  - {_err}")
    raise RuntimeError(f"{len(_errors)} environment check(s) failed — fix them before running any experiment cell.")

print("\n[OK] environment sanity check passed. You may run the experiment cells below.")


## Seed 0 — all 27 UNM experiments


In [ ]:
# === RUN 1/108: supervised/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 2/108: semi_r3/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r3"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r3_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 3/108: semi_r5/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r5"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r5_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 4/108: semi_r7/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r7"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r7_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 5/108: semi_r10/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 6/108: semi_r15/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r15"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r15_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 7/108: semi_r20/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r20"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r20_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 8/108: semi_std_matched_r3/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r3"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r3/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 9/108: semi_std_matched_r5/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r5"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r5/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 10/108: semi_std_matched_r7/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r7"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r7/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 11/108: semi_std_matched_r10/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r10"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 12/108: semi_std_matched_r15/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r15"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r15/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 13/108: semi_std_matched_r20/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r20"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r20/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 14/108: semi_all_lateral/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_all_lateral"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 15/108: mean_teacher_r3/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r3"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r3_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 16/108: mean_teacher_r5/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r5"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r5_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 17/108: mean_teacher_r7/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r7"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r7_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 18/108: mean_teacher_r10/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 19/108: mean_teacher_r15/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r15"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r15_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 20/108: mean_teacher_r20/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r20"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r20_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 21/108: mean_teacher_std_matched_r3/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r3"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r3/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 22/108: mean_teacher_std_matched_r5/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r5"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r5/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 23/108: mean_teacher_std_matched_r7/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r7"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r7/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 24/108: mean_teacher_std_matched_r10/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r10"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 25/108: mean_teacher_std_matched_r15/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r15"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r15/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 26/108: mean_teacher_std_matched_r20/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r20"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r20/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 27/108: mean_teacher_all_lateral/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 28/108: supervised_bifpn_unet/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_bifpn_unet"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: BiFPN-U-Net(T) VGG16 random init
cfg["arch"]      = "bifpn_unet"
cfg["backbone"]  = "vgg16"
cfg["n_classes"] = 1

# Supervised only (BiFPN-U-Net(T) VGG16 random init baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 29/108: supervised_unet/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_unet"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: U-Net
cfg["arch"]      = "unet"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (U-Net baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 30/108: supervised_deeplabv3plus/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_deeplabv3plus"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: DeepLabV3+
cfg["arch"]      = "deeplabv3plus"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (DeepLabV3+ baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 31/108: supervised_fpn/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_fpn"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: FPN
cfg["arch"]      = "fpn"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (FPN baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 32/108: supervised_transunet/seed_0 ===
import os
os.environ["TRANSUNET_PRETRAINED_PATH"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/R50+ViT-B_16.npz"

import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_transunet"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: TransUNet (R50+ViT-B_16)
cfg["arch"]      = "transunet"
cfg["backbone"]  = "R50-ViT-B_16"
cfg["n_classes"] = 1

# Supervised only (TransUNet (R50+ViT-B_16) baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 33/108: supervised_frac25/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_frac25"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (frac25 label subset, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 34/108: supervised_frac50/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_frac50"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (frac50 label subset, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 35/108: supervised_frac75/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_frac75"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (frac75 label subset, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 36/108: semi_r10_frac25/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_frac25"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac25 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 37/108: semi_r10_frac50/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_frac50"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac50 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 38/108: semi_r10_frac75/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_frac75"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac75 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === SEED 0 SUMMARY (UNM) ===
# READ-ONLY. No training, no file writes. Scans the output root for all
# runs of seed 0 and prints a compact diagnostic table.
# Fallback: if *_run_report.json is missing, tries to read test F1 from
# test_metrics.csv directly. Never crashes on a missing file.
import os, json, glob, csv as _csv_mod

_SEED = 0
_ROOT = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1"
_SEMI_START = 15
_EXPECTED = [
    "supervised",
    "semi_r3",
    "semi_r5",
    "semi_r7",
    "semi_r10",
    "semi_r15",
    "semi_r20",
    "semi_std_matched_r3",
    "semi_std_matched_r5",
    "semi_std_matched_r7",
    "semi_std_matched_r10",
    "semi_std_matched_r15",
    "semi_std_matched_r20",
    "semi_all_lateral",
    "mean_teacher_r3",
    "mean_teacher_r5",
    "mean_teacher_r7",
    "mean_teacher_r10",
    "mean_teacher_r15",
    "mean_teacher_r20",
    "mean_teacher_std_matched_r3",
    "mean_teacher_std_matched_r5",
    "mean_teacher_std_matched_r7",
    "mean_teacher_std_matched_r10",
    "mean_teacher_std_matched_r15",
    "mean_teacher_std_matched_r20",
    "mean_teacher_all_lateral",
    "supervised_bifpn_unet",
    "supervised_unet",
    "supervised_deeplabv3plus",
    "supervised_fpn",
    "supervised_transunet",
    "supervised_frac25",
    "supervised_frac50",
    "supervised_frac75",
    "semi_r10_frac25",
    "semi_r10_frac50",
    "semi_r10_frac75",
]

_rows = []
_n_complete = 0
_n_partial = 0
_n_missing = 0
_n_pre_ssl = 0

for _name in _EXPECTED:
    _dir = os.path.join(_ROOT, _name, f"seed_{_SEED}")
    _best = os.path.join(_dir, "best_model.pt")
    _metrics = os.path.join(_dir, "test_metrics.csv")
    _reports = sorted(glob.glob(os.path.join(_dir, "*_run_report.json")))

    if os.path.isfile(_best) and os.path.isfile(_metrics) and _reports:
        _status = "complete"
        _n_complete += 1
    elif os.path.isfile(_best):
        _status = "partial"
        _n_partial += 1
    else:
        _status = "missing"
        _n_missing += 1
        _rows.append((_name, _status, None, None, None))
        continue

    _best_epoch = None
    _is_post_ssl = None
    _test_f1 = None

    # Primary: run_report.json
    if _reports:
        try:
            with open(_reports[0]) as _f:
                _rpt = json.load(_f)
            _ts = (_rpt.get("training_summary") or {})
            _best_epoch = _ts.get("best_epoch")
            _tm = (_rpt.get("test_metrics") or {})
            _test_f1 = _tm.get("sample_mean_f1") or _tm.get("f1_mean")
            _cfg = (_rpt.get("config") or {})
            _use_semi = _cfg.get("use_semi", False)
            if _use_semi and isinstance(_best_epoch, (int, float)):
                _is_post_ssl = _best_epoch >= _SEMI_START
                if not _is_post_ssl:
                    _n_pre_ssl += 1
            else:
                _is_post_ssl = None
        except Exception:
            pass

    # Fallback: test_metrics.csv (if run_report missing or unreadable)
    if _test_f1 is None and os.path.isfile(_metrics):
        try:
            with open(_metrics) as _f:
                for _row in _csv_mod.DictReader(_f):
                    _k = _row.get("metric")
                    if _k in ("sample_mean_f1", "f1_mean"):
                        _test_f1 = float(_row["value"])
                        break
        except Exception:
            pass

    _rows.append((_name, _status, _best_epoch, _is_post_ssl, _test_f1))

# Print compact table
print(f"=== SEED {_SEED} SUMMARY (UNM) — expected {len(_EXPECTED)} runs ===\n")
print(f"{'experiment'.ljust(40)} {'status'.ljust(9)} {'best_ep'.rjust(8)} {'post-SSL'.rjust(9)} {'test F1'.rjust(9)}")
print("-" * 80)
for _name, _status, _best_ep, _post, _f1 in _rows:
    _ep_s = f"{_best_ep}" if _best_ep is not None else "-"
    _post_s = ("yes" if _post else "no") if _post is not None else "-"
    _f1_s = f"{_f1:.4f}" if isinstance(_f1, (int, float)) else "-"
    print(f"{_name[:40].ljust(40)} {_status.ljust(9)} {_ep_s.rjust(8)} {_post_s.rjust(9)} {_f1_s.rjust(9)}")

print("-" * 80)
print(f"totals: {_n_complete} complete, {_n_partial} partial, {_n_missing} missing, {_n_pre_ssl} pre-SSL (best_epoch < semi_start_epoch={_SEMI_START})")


## Seed 1 — all 27 UNM experiments


In [ ]:
# === RUN 39/108: supervised/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 40/108: semi_r3/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r3"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r3_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 41/108: semi_r5/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r5"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r5_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 42/108: semi_r7/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r7"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r7_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 43/108: semi_r10/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 44/108: semi_r15/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r15"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r15_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 45/108: semi_r20/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r20"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r20_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 46/108: semi_std_matched_r3/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r3"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r3/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 47/108: semi_std_matched_r5/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r5"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r5/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 48/108: semi_std_matched_r7/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r7"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r7/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 49/108: semi_std_matched_r10/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r10"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 50/108: semi_std_matched_r15/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r15"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r15/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 51/108: semi_std_matched_r20/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r20"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r20/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 52/108: semi_all_lateral/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_all_lateral"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 53/108: mean_teacher_r3/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r3"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r3_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 54/108: mean_teacher_r5/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r5"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r5_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 55/108: mean_teacher_r7/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r7"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r7_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 56/108: mean_teacher_r10/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 57/108: mean_teacher_r15/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r15"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r15_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 58/108: mean_teacher_r20/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r20"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r20_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 59/108: mean_teacher_std_matched_r3/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r3"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r3/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 60/108: mean_teacher_std_matched_r5/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r5"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r5/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 61/108: mean_teacher_std_matched_r7/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r7"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r7/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 62/108: mean_teacher_std_matched_r10/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r10"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 63/108: mean_teacher_std_matched_r15/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r15"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r15/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 64/108: mean_teacher_std_matched_r20/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r20"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r20/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 65/108: mean_teacher_all_lateral/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 66/108: supervised_bifpn_unet/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_bifpn_unet"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: BiFPN-U-Net(T) VGG16 random init
cfg["arch"]      = "bifpn_unet"
cfg["backbone"]  = "vgg16"
cfg["n_classes"] = 1

# Supervised only (BiFPN-U-Net(T) VGG16 random init baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 67/108: supervised_unet/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_unet"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: U-Net
cfg["arch"]      = "unet"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (U-Net baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 68/108: supervised_frac25/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_frac25"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (frac25 label subset, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 69/108: supervised_frac50/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_frac50"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (frac50 label subset, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 70/108: supervised_frac75/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_frac75"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (frac75 label subset, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 71/108: semi_r10_frac25/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_frac25"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac25 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 72/108: semi_r10_frac50/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_frac50"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac50 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 73/108: semi_r10_frac75/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_frac75"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac75 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === SEED 1 SUMMARY (UNM) ===
# READ-ONLY. No training, no file writes. Scans the output root for all
# runs of seed 1 and prints a compact diagnostic table.
# Fallback: if *_run_report.json is missing, tries to read test F1 from
# test_metrics.csv directly. Never crashes on a missing file.
import os, json, glob, csv as _csv_mod

_SEED = 1
_ROOT = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1"
_SEMI_START = 15
_EXPECTED = [
    "supervised",
    "semi_r3",
    "semi_r5",
    "semi_r7",
    "semi_r10",
    "semi_r15",
    "semi_r20",
    "semi_std_matched_r3",
    "semi_std_matched_r5",
    "semi_std_matched_r7",
    "semi_std_matched_r10",
    "semi_std_matched_r15",
    "semi_std_matched_r20",
    "semi_all_lateral",
    "mean_teacher_r3",
    "mean_teacher_r5",
    "mean_teacher_r7",
    "mean_teacher_r10",
    "mean_teacher_r15",
    "mean_teacher_r20",
    "mean_teacher_std_matched_r3",
    "mean_teacher_std_matched_r5",
    "mean_teacher_std_matched_r7",
    "mean_teacher_std_matched_r10",
    "mean_teacher_std_matched_r15",
    "mean_teacher_std_matched_r20",
    "mean_teacher_all_lateral",
    "supervised_bifpn_unet",
    "supervised_unet",
    "supervised_frac25",
    "supervised_frac50",
    "supervised_frac75",
    "semi_r10_frac25",
    "semi_r10_frac50",
    "semi_r10_frac75",
]

_rows = []
_n_complete = 0
_n_partial = 0
_n_missing = 0
_n_pre_ssl = 0

for _name in _EXPECTED:
    _dir = os.path.join(_ROOT, _name, f"seed_{_SEED}")
    _best = os.path.join(_dir, "best_model.pt")
    _metrics = os.path.join(_dir, "test_metrics.csv")
    _reports = sorted(glob.glob(os.path.join(_dir, "*_run_report.json")))

    if os.path.isfile(_best) and os.path.isfile(_metrics) and _reports:
        _status = "complete"
        _n_complete += 1
    elif os.path.isfile(_best):
        _status = "partial"
        _n_partial += 1
    else:
        _status = "missing"
        _n_missing += 1
        _rows.append((_name, _status, None, None, None))
        continue

    _best_epoch = None
    _is_post_ssl = None
    _test_f1 = None

    # Primary: run_report.json
    if _reports:
        try:
            with open(_reports[0]) as _f:
                _rpt = json.load(_f)
            _ts = (_rpt.get("training_summary") or {})
            _best_epoch = _ts.get("best_epoch")
            _tm = (_rpt.get("test_metrics") or {})
            _test_f1 = _tm.get("sample_mean_f1") or _tm.get("f1_mean")
            _cfg = (_rpt.get("config") or {})
            _use_semi = _cfg.get("use_semi", False)
            if _use_semi and isinstance(_best_epoch, (int, float)):
                _is_post_ssl = _best_epoch >= _SEMI_START
                if not _is_post_ssl:
                    _n_pre_ssl += 1
            else:
                _is_post_ssl = None
        except Exception:
            pass

    # Fallback: test_metrics.csv (if run_report missing or unreadable)
    if _test_f1 is None and os.path.isfile(_metrics):
        try:
            with open(_metrics) as _f:
                for _row in _csv_mod.DictReader(_f):
                    _k = _row.get("metric")
                    if _k in ("sample_mean_f1", "f1_mean"):
                        _test_f1 = float(_row["value"])
                        break
        except Exception:
            pass

    _rows.append((_name, _status, _best_epoch, _is_post_ssl, _test_f1))

# Print compact table
print(f"=== SEED {_SEED} SUMMARY (UNM) — expected {len(_EXPECTED)} runs ===\n")
print(f"{'experiment'.ljust(40)} {'status'.ljust(9)} {'best_ep'.rjust(8)} {'post-SSL'.rjust(9)} {'test F1'.rjust(9)}")
print("-" * 80)
for _name, _status, _best_ep, _post, _f1 in _rows:
    _ep_s = f"{_best_ep}" if _best_ep is not None else "-"
    _post_s = ("yes" if _post else "no") if _post is not None else "-"
    _f1_s = f"{_f1:.4f}" if isinstance(_f1, (int, float)) else "-"
    print(f"{_name[:40].ljust(40)} {_status.ljust(9)} {_ep_s.rjust(8)} {_post_s.rjust(9)} {_f1_s.rjust(9)}")

print("-" * 80)
print(f"totals: {_n_complete} complete, {_n_partial} partial, {_n_missing} missing, {_n_pre_ssl} pre-SSL (best_epoch < semi_start_epoch={_SEMI_START})")


## Seed 2 — all 27 UNM experiments


In [ ]:
# === RUN 74/108: supervised/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 75/108: semi_r3/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r3"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r3_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 76/108: semi_r5/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r5"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r5_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 77/108: semi_r7/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r7"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r7_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 78/108: semi_r10/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 79/108: semi_r15/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r15"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r15_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 80/108: semi_r20/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r20"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r20_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 81/108: semi_std_matched_r3/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r3"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r3/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 82/108: semi_std_matched_r5/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r5"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r5/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 83/108: semi_std_matched_r7/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r7"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r7/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 84/108: semi_std_matched_r10/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r10"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 85/108: semi_std_matched_r15/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r15"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r15/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 86/108: semi_std_matched_r20/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_std_matched_r20"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r20/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 87/108: semi_all_lateral/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_all_lateral"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 88/108: mean_teacher_r3/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r3"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r3_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 89/108: mean_teacher_r5/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r5"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r5_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 90/108: mean_teacher_r7/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r7"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r7_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 91/108: mean_teacher_r10/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 92/108: mean_teacher_r15/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r15"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r15_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 93/108: mean_teacher_r20/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r20"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_r20_max0/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 94/108: mean_teacher_std_matched_r3/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r3"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r3/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 95/108: mean_teacher_std_matched_r5/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r5"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r5/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 96/108: mean_teacher_std_matched_r7/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r7"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r7/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 97/108: mean_teacher_std_matched_r10/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r10"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 98/108: mean_teacher_std_matched_r15/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r15"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r15/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 99/108: mean_teacher_std_matched_r20/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_std_matched_r20"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r20/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 100/108: mean_teacher_all_lateral/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (path from historical 01_train_eval_colab.ipynb)
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 101/108: supervised_bifpn_unet/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_bifpn_unet"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: BiFPN-U-Net(T) VGG16 random init
cfg["arch"]      = "bifpn_unet"
cfg["backbone"]  = "vgg16"
cfg["n_classes"] = 1

# Supervised only (BiFPN-U-Net(T) VGG16 random init baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 102/108: supervised_unet/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_unet"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: U-Net
cfg["arch"]      = "unet"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (U-Net baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 103/108: supervised_frac25/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_frac25"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (frac25 label subset, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 104/108: supervised_frac50/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_frac50"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (frac50 label subset, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 105/108: supervised_frac75/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_frac75"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (frac75 label subset, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 106/108: semi_r10_frac25/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_frac25"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac25 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 107/108: semi_r10_frac50/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_frac50"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac50 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 108/108: semi_r10_frac75/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_frac75"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac75 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === SEED 2 SUMMARY (UNM) ===
# READ-ONLY. No training, no file writes. Scans the output root for all
# runs of seed 2 and prints a compact diagnostic table.
# Fallback: if *_run_report.json is missing, tries to read test F1 from
# test_metrics.csv directly. Never crashes on a missing file.
import os, json, glob, csv as _csv_mod

_SEED = 2
_ROOT = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1"
_SEMI_START = 15
_EXPECTED = [
    "supervised",
    "semi_r3",
    "semi_r5",
    "semi_r7",
    "semi_r10",
    "semi_r15",
    "semi_r20",
    "semi_std_matched_r3",
    "semi_std_matched_r5",
    "semi_std_matched_r7",
    "semi_std_matched_r10",
    "semi_std_matched_r15",
    "semi_std_matched_r20",
    "semi_all_lateral",
    "mean_teacher_r3",
    "mean_teacher_r5",
    "mean_teacher_r7",
    "mean_teacher_r10",
    "mean_teacher_r15",
    "mean_teacher_r20",
    "mean_teacher_std_matched_r3",
    "mean_teacher_std_matched_r5",
    "mean_teacher_std_matched_r7",
    "mean_teacher_std_matched_r10",
    "mean_teacher_std_matched_r15",
    "mean_teacher_std_matched_r20",
    "mean_teacher_all_lateral",
    "supervised_bifpn_unet",
    "supervised_unet",
    "supervised_frac25",
    "supervised_frac50",
    "supervised_frac75",
    "semi_r10_frac25",
    "semi_r10_frac50",
    "semi_r10_frac75",
]

_rows = []
_n_complete = 0
_n_partial = 0
_n_missing = 0
_n_pre_ssl = 0

for _name in _EXPECTED:
    _dir = os.path.join(_ROOT, _name, f"seed_{_SEED}")
    _best = os.path.join(_dir, "best_model.pt")
    _metrics = os.path.join(_dir, "test_metrics.csv")
    _reports = sorted(glob.glob(os.path.join(_dir, "*_run_report.json")))

    if os.path.isfile(_best) and os.path.isfile(_metrics) and _reports:
        _status = "complete"
        _n_complete += 1
    elif os.path.isfile(_best):
        _status = "partial"
        _n_partial += 1
    else:
        _status = "missing"
        _n_missing += 1
        _rows.append((_name, _status, None, None, None))
        continue

    _best_epoch = None
    _is_post_ssl = None
    _test_f1 = None

    # Primary: run_report.json
    if _reports:
        try:
            with open(_reports[0]) as _f:
                _rpt = json.load(_f)
            _ts = (_rpt.get("training_summary") or {})
            _best_epoch = _ts.get("best_epoch")
            _tm = (_rpt.get("test_metrics") or {})
            _test_f1 = _tm.get("sample_mean_f1") or _tm.get("f1_mean")
            _cfg = (_rpt.get("config") or {})
            _use_semi = _cfg.get("use_semi", False)
            if _use_semi and isinstance(_best_epoch, (int, float)):
                _is_post_ssl = _best_epoch >= _SEMI_START
                if not _is_post_ssl:
                    _n_pre_ssl += 1
            else:
                _is_post_ssl = None
        except Exception:
            pass

    # Fallback: test_metrics.csv (if run_report missing or unreadable)
    if _test_f1 is None and os.path.isfile(_metrics):
        try:
            with open(_metrics) as _f:
                for _row in _csv_mod.DictReader(_f):
                    _k = _row.get("metric")
                    if _k in ("sample_mean_f1", "f1_mean"):
                        _test_f1 = float(_row["value"])
                        break
        except Exception:
            pass

    _rows.append((_name, _status, _best_epoch, _is_post_ssl, _test_f1))

# Print compact table
print(f"=== SEED {_SEED} SUMMARY (UNM) — expected {len(_EXPECTED)} runs ===\n")
print(f"{'experiment'.ljust(40)} {'status'.ljust(9)} {'best_ep'.rjust(8)} {'post-SSL'.rjust(9)} {'test F1'.rjust(9)}")
print("-" * 80)
for _name, _status, _best_ep, _post, _f1 in _rows:
    _ep_s = f"{_best_ep}" if _best_ep is not None else "-"
    _post_s = ("yes" if _post else "no") if _post is not None else "-"
    _f1_s = f"{_f1:.4f}" if isinstance(_f1, (int, float)) else "-"
    print(f"{_name[:40].ljust(40)} {_status.ljust(9)} {_ep_s.rjust(8)} {_post_s.rjust(9)} {_f1_s.rjust(9)}")

print("-" * 80)
print(f"totals: {_n_complete} complete, {_n_partial} partial, {_n_missing} missing, {_n_pre_ssl} pre-SSL (best_epoch < semi_start_epoch={_SEMI_START})")


## Architecture comparison — additional seeds (1, 2)


In [ ]:
# === RUN 109/114: supervised_deeplabv3plus/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_deeplabv3plus"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: DeepLabV3+
cfg["arch"]      = "deeplabv3plus"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (DeepLabV3+ baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 110/114: supervised_deeplabv3plus/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_deeplabv3plus"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: DeepLabV3+
cfg["arch"]      = "deeplabv3plus"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (DeepLabV3+ baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 111/114: supervised_fpn/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_fpn"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: FPN
cfg["arch"]      = "fpn"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (FPN baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 112/114: supervised_fpn/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_fpn"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: FPN
cfg["arch"]      = "fpn"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (FPN baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 113/114: supervised_transunet/seed_1 ===
import os
os.environ["TRANSUNET_PRETRAINED_PATH"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/R50+ViT-B_16.npz"

import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_transunet"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: TransUNet (R50+ViT-B_16)
cfg["arch"]      = "transunet"
cfg["backbone"]  = "R50-ViT-B_16"
cfg["n_classes"] = 1

# Supervised only (TransUNet (R50+ViT-B_16) baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 114/114: supervised_transunet/seed_2 ===
import os
os.environ["TRANSUNET_PRETRAINED_PATH"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/R50+ViT-B_16.npz"

import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_transunet"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture: TransUNet (R50+ViT-B_16)
cfg["arch"]      = "transunet"
cfg["backbone"]  = "R50-ViT-B_16"
cfg["n_classes"] = 1

# Supervised only (TransUNet (R50+ViT-B_16) baseline, final schedule patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Mean Teacher label-efficiency experiments (UNM, r=10)


In [ ]:
# === RUN 115/123: mean_teacher_r10_frac25/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_frac25"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac25 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 116/123: mean_teacher_r10_frac25/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_frac25"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac25 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 117/123: mean_teacher_r10_frac25/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_frac25"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac25 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 118/123: mean_teacher_r10_frac50/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_frac50"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac50 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 119/123: mean_teacher_r10_frac50/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_frac50"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac50 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 120/123: mean_teacher_r10_frac50/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_frac50"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac50 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 121/123: mean_teacher_r10_frac75/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_frac75"
_SEED     = 0
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac75 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 122/123: mean_teacher_r10_frac75/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_frac75"
_SEED     = 1
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac75 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 123/123: mean_teacher_r10_frac75/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_frac75"
_SEED     = 2
# Final rerun output root (separate from historical runs/ and runs_lambda_ablation/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (U-Net++ / efficientnet-b3, same as main supervised)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised frac75 (final config: lambda_u=0.05, semi_start=15, patience=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM final schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # final schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool (PL temporal r=10, same path as historical 01)
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Labeled subset file (path from historical 01_train_eval_colab.ipynb)
cfg["labeled_subset_file"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt"

# Final rerun: disable automatic visualization (use post-hoc viz cell)
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Post-hoc visualization

Edit `EXP_NAME` and `SEED` and re-run to generate figures for a completed run.


In [ ]:
# === POST-HOC VISUALIZATION ===
# Edit RUN_ROOT, EXP_NAME, SEED below and re-run this cell to generate
# prediction figures for an existing trained run. Does NOT retrain.

RUN_ROOT = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1"
EXP_NAME = "semi_r10"   # <-- EDIT
SEED = 0                         # <-- EDIT

import os, json, torch
from src.models import create_model
from src.augmentations import get_supervised_train_augmentation
from src.datasets import build_supervised_datasets, build_dataloaders
from src.evaluate import evaluate_checkpoint

_exp_dir = f"{RUN_ROOT}/{EXP_NAME}/seed_{SEED}"
_cfg_path = os.path.join(_exp_dir, "config.json")
with open(_cfg_path) as f:
    cfg = json.load(f)
# Force viz ON only for this post-hoc pass
cfg["save_preds_vis"] = True
cfg["exp_dir"] = _exp_dir

train_tf = get_supervised_train_augmentation(cfg)
_, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
loaders = build_dataloaders(cfg, train_ds=None, val_ds=val_ds, test_ds=test_ds,
                             unlabeled_ds=None, temporal_unlab_ds=None)

model = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
_best_path = os.path.join(_exp_dir, "best_model.pt")
results = evaluate_checkpoint(cfg, model, loaders, _best_path, [])
print(f"Visualization saved to: {os.path.join(_exp_dir, 'preds_vis')}")
